# Examples of Using `nomis` API

Load in the API wrapper and generate an index of the available datasets:

In [2]:
from nomis_api_wrapper import NOMIS_CONFIG

nomis = NOMIS_CONFIG()

nomis_datasets = nomis.nomis_codes_datasets()
nomis_datasets.head()

,agency,idx,name,description,source
0,NOMIS,NM_1_1,Jobseeker's Allowance with rates and proportions,Records the number of people claiming Jobseeke...,jsa
1,NOMIS,NM_2_1,claimant count - age and duration,A quartery count of claimants who were claimin...,jsa
2,NOMIS,NM_4_1,Jobseeker's Allowance by age and duration,A monthly count of Jobseeker's Allowance (JSA)...,jsa
3,NOMIS,NM_5_1,claimant count denominators - historical workf...,A midyear estimate of the workforce (the denom...,jsa
4,NOMIS,NM_6_1,claimant count - occupation,A quarterly count of Jobseeker's Allowance cl...,jsa


## Census Datasets

Census data is available via `nomis`.

In [3]:
census_datasets = nomis_datasets[nomis_datasets["source"].str.contains("census_2021")]
census_datasets.head()

,agency,idx,name,description,source
1291,NOMIS,NM_2020_1,TS007A - Age by five-year age bands,This dataset provides Census 2021 estimates th...,census_2021_ts
1292,NOMIS,NM_2021_1,TS001 - Number of usual residents in household...,This dataset provides Census 2021 estimates th...,census_2021_ts
1293,NOMIS,NM_2022_1,TS002 - Legal partnership status,This dataset provides Census 2021 estimates th...,census_2021_ts
1294,NOMIS,NM_2023_1,TS003 - Household composition,This dataset provides Census 2021 estimates th...,census_2021_ts
1295,NOMIS,NM_2024_1,TS004 - Country of birth,This dataset provides Census 2021 estimates th...,census_2021_ts


For example, the *Number of usual residents* gives us example population numbers.

In [4]:
USUAL_RESIDENTS_2021 = census_datasets[
    census_datasets["name"].str.contains("usual residents in households")
].reset_index().loc[0, "idx"]

USUAL_RESIDENTS_2021

'NM_2021_1'

In [5]:
NM_2021_1_metadata = nomis.nomis_code_metadata(USUAL_RESIDENTS_2021)
NM_2021_1_metadata

{'core':   agency        idx                                               name  \
 0  NOMIS  NM_2021_1  TS001 - Number of usual residents in household...   
 1  NOMIS  NM_2021_1  TS001 - Number of usual residents in household...   
 2  NOMIS  NM_2021_1  TS001 - Number of usual residents in household...   
 3  NOMIS  NM_2021_1  TS001 - Number of usual residents in household...   
 
                                          description          source  \
 0  This dataset provides Census 2021 estimates th...  census_2021_ts   
 1  This dataset provides Census 2021 estimates th...  census_2021_ts   
 2  This dataset provides Census 2021 estimates th...  census_2021_ts   
 3  This dataset provides Census 2021 estimates th...  census_2021_ts   
 
                    dimension          concept  
 0        CL_2021_1_GEOGRAPHY        GEOGRAPHY  
 1  CL_2021_1_C2021_RESTYPE_3  C2021_RESTYPE_3  
 2         CL_2021_1_MEASURES         MEASURES  
 3             CL_2021_1_FREQ             FREQ  ,
 '

The results are provided for specified geographical areas. We can view the top level geographies for which the data is available.

In [6]:
# USUAL_RESIDENTS_2021 = "NM_2021_1"

# Get the geography codes for which this data is available
residents_geog_codes = nomis.nomis_codes_geog(USUAL_RESIDENTS_2021)
residents_geog_codes

,agencyid,dataset,dimension,name,description,value
0,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,England and Wales,2092957703
1,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,England,2092957699
2,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Wales,2092957700


For a particular geography code supported by a dataset, we can review the available datasets (lower level geographies) within that high-level geography:

In [7]:
geog_code = residents_geog_codes[residents_geog_codes["description"] == "England"]["value"].iloc[0]

england_geog = nomis.nomis_codes_geog(USUAL_RESIDENTS_2021, geography=geog_code)
england_geog.head(10)

,agencyid,dataset,dimension,name,description,value
0,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,England,2092957699
1,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 output areas within England,2092957699TYPE150
2,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 super output areas - lower layer within E...,2092957699TYPE151
3,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 super output areas - middle layer within ...,2092957699TYPE152
4,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2022 wards within England,2092957699TYPE153
5,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2022 local authorities: districts within England,2092957699TYPE154
6,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2022 local authorities: counties within England,2092957699TYPE155
7,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 Parliamentary constituencies within England,2092957699TYPE158
8,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Sub Integrated Care Board Locations within Eng...,2092957699TYPE164
9,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Integrated Care Boards within England,2092957699TYPE166


Let's see if the data is available grouped by LA districts:

In [8]:
en_la_districts = england_geog[england_geog["description"].str.contains("districts")].to_dict(orient="records")[0]
en_la_districts

{'agencyid': 'NOMIS',
 'dataset': 'NM_2021_1',
 'dimension': 'CL_2021_1_GEOGRAPHY',
 'name': 'geography',
 'description': '2022 local authorities: districts within England',
 'value': '2092957699TYPE154'}

See if we can find the geography code for the *Isle of Wight* LAD. We need to search in the *2022 local authorities: districts within England* geography, whose identifier is given by the `CL_2021_1_GEOGRAPHY` dimension `value`.

In [11]:
iw = nomis.nomis_codes_geog(
    USUAL_RESIDENTS_2021,
    geography=en_la_districts["value"],
    search="Isle of Wight*",
).to_dict(orient="records")[0]

iw

{'agencyid': 'NOMIS',
 'dataset': 'NM_2021_1',
 'dimension': 'CL_2021_1_GEOGRAPHY',
 'name': 'geography',
 'description': 'Isle of Wight',
 'value': 645923034}

In [12]:
nomis.nomis_codes_geog(
    USUAL_RESIDENTS_2021,
    geography=iw["value"]
)

,agencyid,dataset,dimension,name,description,value
0,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Isle of Wight,645923034
1,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 output areas within Isle of Wight,645923034TYPE150
2,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 super output areas - lower layer within I...,645923034TYPE151
3,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2021 super output areas - middle layer within ...,645923034TYPE152
4,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,2022 wards within Isle of Wight,645923034TYPE153


The rather cryptic `nomis.get_geo_code()` has a chase function that will also try to find corresponding records in child geographies in a rather more convenient way:

In [46]:
iw_ = nomis.get_geo_code(idx=USUAL_RESIDENTS_2021,value=en_la_districts["value"], search="Isle of Wight*", chase=True)
iw_

,description,geog
0,Isle of Wight,645923034
1,2021 output areas within Isle of Wight,645923034TYPE150
2,2021 super output areas - lower layer within I...,645923034TYPE151
3,2021 super output areas - middle layer within ...,645923034TYPE152
4,2022 wards within Isle of Wight,645923034TYPE153


So how does the population break down for the Isle of Wight LAD?

In [ ]:
cols = ["GEOGRAPHY_CODE", "C2021_RESTYPE_3_NAME", "MEASURES_NAME", "OBS_VALUE"]

nomis._nomis_data(USUAL_RESIDENTS_2021, geography=iw_["geog"].iloc[0])[cols]

,GEOGRAPHY_CODE,C2021_RESTYPE_3_NAME,MEASURES_NAME,OBS_VALUE
0,E06000046,Total: All usual residents,Value,140459.0
1,E06000046,Total: All usual residents,Percent,100.0
2,E06000046,Lives in a household,Value,137516.0
3,E06000046,Lives in a household,Percent,97.9
4,E06000046,Lives in a communal establishment,Value,2943.0
5,E06000046,Lives in a communal establishment,Percent,2.1


We can also get the LSOA geography codes:

In [58]:
iw_lower = iw_[iw_["description"].str.contains("lower layer")].to_dict(
    orient="records"
)[0]
iw_lower

{'description': '2021 super output areas - lower layer within Isle of Wight',
 'geog': '645923034TYPE151'}

In [68]:
iw_lower_areas = nomis.nomis_codes_geog(
    USUAL_RESIDENTS_2021, geography=iw_lower["geog"]
)
iw_lower_areas.head()

,agencyid,dataset,dimension,name,description,value
0,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Isle of Wight 001A,633356334
1,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Isle of Wight 001B,633356337
2,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Isle of Wight 001C,633356339
3,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Isle of Wight 001D,633356340
4,NOMIS,NM_2021_1,CL_2021_1_GEOGRAPHY,geography,Isle of Wight 001E,633356343


We can now query the data values by LSOA area code:

In [ ]:
# Resident population count in a single LSOA
nomis._nomis_data(USUAL_RESIDENTS_2021, geography=iw_lower_areas["value"].iloc[0])[cols]

,GEOGRAPHY_CODE,C2021_RESTYPE_3_NAME,MEASURES_NAME,OBS_VALUE
0,E01017300,Total: All usual residents,Value,2151.0
1,E01017300,Total: All usual residents,Percent,100.0
2,E01017300,Lives in a household,Value,2095.0
3,E01017300,Lives in a household,Percent,97.4
4,E01017300,Lives in a communal establishment,Value,56.0
5,E01017300,Lives in a communal establishment,Percent,2.6


We can also request data for a multiple geographies. For example, to query over all the LSOAs in the Isle of Wight we can concatenate all the LSOA codes as a comma separated geography value:

In [77]:
iw_lsoa_codes = ",".join(iw_lower_areas["value"].astype(str))
iw_lsoa_pops = nomis._nomis_data(USUAL_RESIDENTS_2021, geography=iw_lsoa_codes)
iw_lsoa_pops.shape

(534, 29)

The raw dataframe is rather messy, but we can reshape it:

In [80]:
def census_data(data_df, obs_col, typ="Value"):
    # typ: Value, Percent
    _cols = ["DATE", "GEOGRAPHY_NAME", "GEOGRAPHY_CODE", "OBS_VALUE"]
    _cols.append(obs_col)

    data_df = data_df[
        (data_df["OBS_STATUS"] != "Q") & (data_df["MEASURES_NAME"] == typ)
    ].copy()

    data_df_wide = (
        data_df[_cols]
        .pivot_table(
            index=["DATE", "GEOGRAPHY_NAME", "GEOGRAPHY_CODE"],
            columns=[obs_col],
            values=["OBS_VALUE"],
        )
        .dropna()
        .reset_index()
    )

    data_df_wide.columns = [
        col[0] if col[1] == "" else col[1] for col in data_df_wide.columns
    ]
    return data_df_wide


In [83]:
census_data(iw_lsoa_pops, "C2021_RESTYPE_3_NAME").head()

,DATE,GEOGRAPHY_NAME,GEOGRAPHY_CODE,Lives in a communal establishment,Lives in a household,Total: All usual residents
0,2021,Isle of Wight 001A,E01017300,56.0,2095.0,2151.0
1,2021,Isle of Wight 001B,E01017303,1.0,1185.0,1186.0
2,2021,Isle of Wight 001C,E01017305,31.0,1493.0,1524.0
3,2021,Isle of Wight 001D,E01017306,0.0,1368.0,1368.0
4,2021,Isle of Wight 001E,E01017309,0.0,1571.0,1571.0
